## SILVER LAYVER - ENRICHED

In [0]:
%run ./00_config

### Download NYC Taxi Zone Lookup

In [0]:
import urllib.request

ZONE_URL    = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
ZONE_VOLUME = f"/Volumes/nyc_taxi_project/landing/raw_files/taxi_zone_lookup.csv"

# Download trực tiếp vào Volume
with urllib.request.urlopen(ZONE_URL) as response:
    data = response.read()

with open(ZONE_VOLUME, "wb") as f:
    f.write(data)

### Create Silver Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nyc_taxi_project.silver.yellow_taxi_silver_enriched
(
  -- Columns từ Silver 1
  VendorID                INT,
  tpep_pickup_datetime    TIMESTAMP,
  tpep_dropoff_datetime   TIMESTAMP,
  passenger_count         INT,
  trip_distance           DOUBLE,
  PULocationID            INT,
  DOLocationID            INT,
  payment_type            INT,
  fare_amount             DOUBLE,
  tip_amount              DOUBLE,
  tolls_amount            DOUBLE,
  total_amount            DOUBLE,
  congestion_surcharge    DOUBLE,
  pickup_year             INT,
  pickup_month            INT,
  pickup_day              INT,
  pickup_hour             INT,
  trip_duration_minutes   DOUBLE,
  -- Enriched columns mới
  pickup_zone             STRING,
  pickup_borough          STRING,
  dropoff_zone            STRING,
  dropoff_borough         STRING,
  payment_type_desc       STRING,
  is_airport_trip         BOOLEAN,
  is_rush_hour            BOOLEAN,
  tip_percentage          DOUBLE,
  -- Metadata
  ingested_at             TIMESTAMP,
  source_file             STRING
)
USING DELTA
COMMENT 'Silver 2 layer - enriched NYC Yellow Taxi data with zone info'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true');

### Transform Function

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def transform_silver2(df):

    # Load zone lookup
    df_zones = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(ZONE_VOLUME))

    # Join pickup zone
    df = (df
        .join(F.broadcast(df_zones.select(
                F.col("LocationID").alias("PULocationID"),
                F.col("Zone").alias("pickup_zone"),
                F.col("Borough").alias("pickup_borough")
              )),
              on="PULocationID", how="left")

        # Join dropoff zone
        .join(F.broadcast(df_zones.select(
                F.col("LocationID").alias("DOLocationID"),
                F.col("Zone").alias("dropoff_zone"),
                F.col("Borough").alias("dropoff_borough")
              )),
              on="DOLocationID", how="left")

        # Payment type description
        .withColumn("payment_type_desc",
            F.when(F.col("payment_type") == 1, "Credit Card")
             .when(F.col("payment_type") == 2, "Cash")
             .when(F.col("payment_type") == 3, "No Charge")
             .when(F.col("payment_type") == 4, "Dispute")
             .otherwise("Unknown"))

        # Airport trip: JFK, LaGuardia, Newark
        .withColumn("is_airport_trip",
            F.col("pickup_zone").rlike("(?i)JFK|LaGuardia|Newark") |
            F.col("dropoff_zone").rlike("(?i)JFK|LaGuardia|Newark"))

        # Rush hour: 7-9am và 5-7pm weekday
        .withColumn("is_rush_hour",
            (F.col("pickup_hour").between(7, 9) |
             F.col("pickup_hour").between(17, 19)) &
            F.dayofweek("tpep_pickup_datetime").between(2, 6))

        # Tip percentage
        .withColumn("tip_percentage",
            F.when(F.col("fare_amount") > 0,
                F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)
            ).otherwise(0.0))
    )

    return df

### Run Silver Enriched Streaming

In [0]:
def process_silver_enriched():
   query = (spark.readStream
               .format("delta")
               .option("ignoreDeletes", "true")
               .table(SILVER1_TABLE)
               .transform(transform_silver2)
            .writeStream
               .option("checkpointLocation", SILVER2_CHECKPOINT)
               .option("mergeSchema", "true")
               .trigger(availableNow=True)
               .table(SILVER2_TABLE))
   query.awaitTermination()
process_silver_enriched()

In [0]:
%sql
ALTER TABLE nyc_taxi_project.silver.yellow_taxi_silver_enriched 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);